In [1]:
from pathlib import Path
import json

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

BASE = Path("/workspaces/MP2")

spark = (
    SparkSession.builder
    .appName("MP5-Reconciliation")
    .master("local[2]")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

HORIZON = "2026-10-21 00:38:00"

KEYS = [
    "window_type",
    "window_start",
    "window_end",
]

LATENCY_TOLERANCE_SEC = 1e-6

def load_results(experiment, query_name):
    path = BASE / "output" / experiment / query_name

    return (
        spark.read.parquet(str(path))
        .filter(
            F.col("window_end")
            <= F.lit(HORIZON).cast("timestamp")
        )
    )

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/21 01:25:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/21 01:25:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/21 01:25:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/10/21 01:25:10 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
def duplicate_key_count(df):
    return (
        df.groupBy(*KEYS)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )


def compare_query(query_name):
    reference = load_results("reference_v1", query_name)
    recovery = load_results("recovery_v1", query_name)

    reference_keys = reference.select(*KEYS).distinct()
    recovery_keys = recovery.select(*KEYS).distinct()

    reference_only = reference_keys.join(
        recovery_keys, KEYS, "left_anti"
    ).count()

    recovery_only = recovery_keys.join(
        reference_keys, KEYS, "left_anti"
    ).count()

    paired = (
        reference.alias("r")
        .join(recovery.alias("s"), KEYS, "inner")
    )

    exact_columns = [
        "record_count",
        "approx_distinct_notes",
        "top_10_notes",
    ]

    mismatch = F.lit(False)

    for column in exact_columns:
        mismatch = mismatch | (
            ~F.col(f"r.{column}").eqNullSafe(
                F.col(f"s.{column}")
            )
        )

    reference_latency = F.col(
        "r.mean_ingest_latency_sec"
    )
    recovery_latency = F.col(
        "s.mean_ingest_latency_sec"
    )

    latency_matches = (
        (
            reference_latency.isNull()
            & recovery_latency.isNull()
        )
        | F.coalesce(
            F.abs(
                reference_latency - recovery_latency
            ) <= LATENCY_TOLERANCE_SEC,
            F.lit(False),
        )
    )

    mismatch = mismatch | (~latency_matches)

    mismatches = paired.filter(mismatch)

    result = {
        "query": query_name,
        "reference_rows": reference.count(),
        "recovery_rows": recovery.count(),
        "reference_duplicate_keys": (
            duplicate_key_count(reference)
        ),
        "recovery_duplicate_keys": (
            duplicate_key_count(recovery)
        ),
        "reference_only_keys": reference_only,
        "recovery_only_keys": recovery_only,
        "aggregate_mismatch_rows": mismatches.count(),
    }

    result["passed"] = (
        result["reference_rows"] == 7
        and result["recovery_rows"] == 7
        and all(
            result[field] == 0
            for field in (
                "reference_duplicate_keys",
                "recovery_duplicate_keys",
                "reference_only_keys",
                "recovery_only_keys",
                "aggregate_mismatch_rows",
            )
        )
    )

    print(json.dumps(result, indent=2))

    if result["aggregate_mismatch_rows"]:
        mismatches.show(truncate=False)

    return result


results = [
    compare_query("tumbling_60s"),
    compare_query("sliding_5m_1m"),
]

{
  "query": "tumbling_60s",
  "reference_rows": 7,
  "recovery_rows": 7,
  "reference_duplicate_keys": 0,
  "recovery_duplicate_keys": 0,
  "reference_only_keys": 0,
  "recovery_only_keys": 0,
  "aggregate_mismatch_rows": 0,
  "passed": true
}


{
  "query": "sliding_5m_1m",
  "reference_rows": 7,
  "recovery_rows": 7,
  "reference_duplicate_keys": 0,
  "recovery_duplicate_keys": 0,
  "reference_only_keys": 0,
  "recovery_only_keys": 0,
  "aggregate_mismatch_rows": 0,
  "passed": true
}


In [3]:
evidence_dir = BASE / "output/reconciliation"
evidence_dir.mkdir(parents=True, exist_ok=True)

summary = {
    "comparison_horizon_utc": HORIZON,
    "latency_tolerance_sec": LATENCY_TOLERANCE_SEC,
    "queries": results,
    "all_passed": all(item["passed"] for item in results),
}

(evidence_dir / "summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

assert summary["all_passed"], (
    "Reconciliation failed. Inspect differences before "
    "claiming successful recovery."
)

In [4]:
spark.stop()